# Validation du model turbulent poreux en TrioCFD - Cas 1

In [ ]:
from trustutils import run 
# import all the packages
from trustutils.jupyter import run
from trustutils import plot, files as tf
from datetime import date

from string import Template 
import re
import numpy as np
import sympy as sp
import pandas as pd
import os
import math
import csv

import matplotlib.pyplot as plt
from IPython.display import display, Markdown as md
run.introduction("ALi Fadlallah")

run.TRUST_parameters()

### Description: 

Run reference cases and the test cases

In [ ]:
# Geometry

# total real dimension of the setup
width = 0.00834 # meter
height = 0.018 # meter

# vertical channel size
channel_width  = 0.00234 # meter
channel_height = 0.006 # meter

In [ ]:
def compute_geometry(porosity, width=0.00834, height=0.018, channel_width=0.00234, channel_height=0.006):
    scorpio_cell_x = round(channel_width / 5, 6)
    mesh_y_scorpio = round(channel_height / 12, 6)

    Lx1 = round(channel_width * (1 - porosity) / 5, 6)
    Lx3 = Lx1
    Lx2 = round(channel_width - (Lx1 + Lx3), 6)
    Lx4 = round(width - (Lx1 + Lx2 + Lx3), 6)

    Ly = round(0.012, 6)
    Ly_PE = round(channel_height / 12, 6)
    Ly_mid = round(channel_height - mesh_y_scorpio, 6)

    # X positions
    X0 = 0.0
    X1 = round(X0 + Lx1, 6)
    X2 = round(X1 + Lx2, 6)
    X3 = round(X2 + Lx3, 6)
    X4 = round(X3 + Lx4, 6)

    # Y positions
    Y0 = round(-Ly, 6)
    Y1 = round(Y0 + Ly, 6)
    Y2 = round(Y1 + Ly_PE, 6)
    Y3 = round(Y2 + Ly_mid, 6)
    Y4 = round(Y3 + Ly, 6)

    return {
        "X0": X0, "X1": X1, "X2": X2, "X3": X3, "X4": X4,
        "Y0": Y0, "Y1": Y1, "Y2": Y2, "Y3": Y3, "Y4": Y4,
        "Lx1": Lx1, "Lx2": Lx2, "Lx3": Lx3, "Lx4": Lx4,
        "Ly": Ly, "Ly_PE": Ly_PE, "Ly_mid": Ly_mid,
    }


In [ ]:
# mesh size for the refernce case
# Node counts
Nx1 = 3
Nx2 = 35 #35
Nx3 = 3
Nx4 = 40 #40
Ny =  160 #160
Ny_PE = 10
Ny_mid = 150 #150



porosities = np.array([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])


run.reset()
run.defaultSuite_ = run.TRUSTSuite(runPrepare=False)

for phi in porosities:
    geometry = compute_geometry(phi)
    geometry.update({"Nx1": Nx1, "Nx2": Nx2, "Nx3": Nx3, "Nx4": Nx4,
                    "Ny": Ny, "Ny_PE": Ny_PE, "Ny_mid": Ny_mid, "channel_width":channel_width, "width": width})
    
    run.addCaseFromTemplate("jdd_ref.data", f"reference/phi_{phi:.1g}", geometry)


run.printCases()

In [ ]:
import itertools


Nx_validation = 30
Ny_validation = 60

#parameters for Frottement_poreux
longueur=0.000936


# Define parameter ranges
c0_values = c0_values = np.array([0.4])
c1_values = np.array([1.0e4])


# Loop over all combinations
for phi, c0, c1 in itertools.product(porosities, c0_values, c1_values):
    #case_name = f"c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1f}"
    
    geometry = compute_geometry(phi)
    run.addCaseFromTemplate(
        "jdd.data", f"validation/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}",
        {   # Frottement poreux
            "longueur": longueur,
            "c_0": c0,
            "c_1": c1,

            "Nx": Nx_validation,
            "Ny": Ny_validation,
            
           
            "porosity": phi,
            "channel_width": channel_width,
            "channel_height": channel_height,
            "width": width,
            "height": height,
            "scorpio_cell_x": round(channel_width / 5, 6),

            # Geometry coordinates
            "X0": geometry["X0"],
            "X1": geometry["X1"],
            "X2": geometry["X2"],
            "X3": geometry["X3"],
            "Y0": geometry["Y0"],
            "Y1": geometry["Y1"],
            "Y2": geometry["Y2"],
            "Y3": geometry["Y3"],
            "Y4": geometry["Y4"],

            # Sizes
            "Lx4": 0.006,
            "Ly": geometry["Ly"],
            "Ly_mid": geometry["Ly_mid"],
            "Ly_PE": geometry["Ly_PE"],

            "solid":1-phi,
        }
    )

run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
def GetSonData(path):
    if not os.path.exists(path):
        raise ValueError(f"The path ({path}) does not exist!")
    # On plot le dernier instant
    donne = tf.SonSEGFile(path, None)
    
    # On recupere le nom des variables, leur dimension et la coordonnee que l'on veut tracer
    ncompo = donne.getnCompo()
    entries = donne.getEntries()
    t = donne.getValues(entries[0])[0]

    ## find closest value of t
    idx = -1
    X = donne.getXAxis()
    Y = np.zeros([len(entries[0::ncompo]), ncompo])
    
    for k in range(ncompo):
        for n, i in enumerate(entries[k::ncompo]):
            Y[n, k] = list(donne.getValues(i)[1])[idx]
    X = np.array(X)
    #print(Y)
    Y = np.array(Y)
    

    return X, Y

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Error computation function using L-norms (normalized by reference)
def compute_L_norm_errors(ref, val):
    epsilon = 1e-10  # To avoid division by zero

    # L1 norm (Mean Relative Error)
    L1_rel = np.mean(np.abs((ref - val) / (ref + epsilon)))

    # L2 norm (Root Mean Square Error, normalized)
    L2_rel = np.sqrt(np.mean(((ref - val) / (ref + epsilon)) ** 2))

    # L-infinity norm (Maximum Relative Error)
    Linf_rel = np.max(np.abs((ref - val) / (ref + epsilon)))

    return L1_rel, L2_rel, Linf_rel

In [ ]:
# Sonde names and labels
sondes_labeled = {"SONDE_PRESSION": "Pressure", "SONDE_VHAUT": "V_haut",
                  "SONDE_VBAS": "V_bas", "SONDE_V_VERTICALE": "V_verticale", "SONDE_VCHANNEL": "V_channel",}

In [ ]:
import itertools
from collections import defaultdict

# Structure: {"SONDE_VBAS" : {(c0, c1): {"L1_rel": [...], "L2_rel": [...], "Linf_rel": [...]}}}
errors_by_case = defaultdict(
    lambda: defaultdict(
        lambda: {"L1_rel": [], "L2_rel": [], "Linf_rel": []}
    )
)
for c0, c1 in itertools.product(c0_values, c1_values):
    for phi in porosities:
        for sonde, label in sondes_labeled.items():
            # File paths
            path_ref = os.path.join(f"{run.BUILD_DIRECTORY}/reference/phi_{phi:.1g}/", f"jdd_ref_{sonde}.son")
            path_val = os.path.join(f"{run.BUILD_DIRECTORY}/validation/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}/", f"jdd_{sonde}.son")
            
            if sonde != 'SONDE_PRESSION' : 
                try:
                    _, V_ref = GetSonData(path_ref)
                    # compute the norm i.e. sqrt(Vx^2 + Vy^2)
                    V_ref = np.linalg.norm(V_ref.reshape(-1, 2), axis=1)
        
                    _, V_val = GetSonData(path_val)
                    V_val = np.linalg.norm(V_val.reshape(-1, 2), axis=1)
                
                except Exception as e:
                    print(f"Error reading or processing: phi={phi}, c0={c0}, c1={c1}, sonde={sonde}\n{e}")
                    continue
                
                # Error computation
                L1_rel, L2_rel, Linf_rel = compute_L_norm_errors(V_ref, V_val)
            else:
                try:
                    _, P_ref = GetSonData(path_ref)
                    _, P_val = GetSonData(path_val)

                    delta_P_ref = np.abs((P_ref[0]- P_ref)/P_ref[0])
                    delta_P_val = np.abs((P_val[0]- P_val)/P_val[0])
                except Exception as e:
                    print(f"Error reading or processing: phi={phi}, c0={c0}, c1={c1}, sonde={sonde}\n{e}")
                    continue
                
                # Error computation
                L1_rel, L2_rel, Linf_rel = compute_L_norm_errors(delta_P_ref, delta_P_val)
            
            # Store with tuple key
            key = (c0, c1)
            errors_by_case[sonde][key]["L1_rel"].append(L1_rel)
            errors_by_case[sonde][key]["L2_rel"].append(L2_rel)
            errors_by_case[sonde][key]["Linf_rel"].append(Linf_rel)

In [ ]:
import pandas as pd
from cycler import cycler
from matplotlib.lines import Line2D

def plot_sonde(sonde):

    c0, c1 = c0_values[0], c1_values[0]   
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(porosities)))
    
    plt.figure(figsize=(10,8), dpi=400)
    plt.grid()
    
    for i, phi in enumerate(porosities):
        # File paths
        path_ref = os.path.join(f"{run.BUILD_DIRECTORY}/reference/phi_{phi:.1g}/", f"jdd_ref_{sonde}.son")
        path_val = os.path.join(f"{run.BUILD_DIRECTORY}/validation/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}/", f"jdd_{sonde}.son")
        
        if sonde == 'SONDE_PRESSION' : 
            Y_ref, P_ref = GetSonData(path_ref)
            Y_val, P_val = GetSonData(path_val)

            P_ref = P_ref[:, 0]
            P_val = P_val[:, 0]
            
            delta_P_ref = np.abs((P_ref[0]- P_ref)/P_ref[0])
            delta_P_val = np.abs((P_val[0]- P_val)/P_val[0])

            plt.plot(Y_ref, delta_P_ref ,marker='*', color=colors[i], label=rf"$\phi_{{ref}}={phi:.1g}$")
            plt.plot(Y_val, delta_P_val ,":", color=colors[i], label=rf"$\phi_{{vald}}={phi:.1g}$")
            
            plt.title(rf"Pressure loss for $c_0={c0:.1g}, c_1={c1:.1g}$")
            plt.ylabel(rf'$\Delta P$ [Pa]')
            plt.xlabel('Y [m]')
            
        else: 
            X, data_ref = GetSonData(path_ref)
            # compute the norm i.e. sqrt(Vx^2 + Vy^2)
            V_ref = np.linalg.norm(data_ref.reshape(-1, 2), axis=1)

            X, data_val = GetSonData(path_val)
            V_val = np.linalg.norm(data_val.reshape(-1, 2), axis=1)

            plt.plot(X, V_ref ,marker='*', color=colors[i], label=rf"$\phi_{{ref}}={phi:.1g}$")
            plt.plot(X, V_val ,":", color=colors[i], label=rf"$\phi_{{vald}}={phi:.1g}$")
            
            # x and y label
            plt.title(rf"{sondes_labeled[sonde]} for $c_0={c0:.1g}, c_1={c1:.1g}$")
            plt.xlabel('X [m]')
            plt.ylabel('V [m/s]')
        plt.ticklabel_format(useOffset=False,style='plain')
        plt.legend()
    plt.tight_layout()
    #plt.savefig(f"sonde_{sondes_labeled[sonde]}.jpg", dpi=400)
    plt.show()

def plot_error_norm(sonde):
  
    plt.figure(figsize=(10, 8))
    plt.plot(1 - porosities, errors_by_case[sonde][(c0, c1)]["L1_rel"],
             label=r"$L_1$", marker='o')
    plt.plot(1 - porosities, errors_by_case[sonde][(c0, c1)]["L2_rel"],
             label=r"$L_2$", marker='s')
    plt.plot(1 - porosities, errors_by_case[sonde][(c0, c1)]["Linf_rel"],
             label=r"$L_\infty$", marker='^')
    
    plt.yscale('log')
    plt.xlabel(r"$1 - \phi$")
    plt.ylabel("Normalized Error")
    plt.title(f"L-norm Errors for {sondes_labeled[sonde]} for c0={c0:.1g}, c1={c1:.1g}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    #plt.savefig(f"L_norms_{sondes_labeled[sonde]}.jpg")
    plt.show()

In [ ]:
for sonde, label in sondes_labeled.items():
    plot_sonde(sonde)
    plot_error_norm(sonde)

In [ ]:
# compute Coefficient de frottement
def coeff_frott(delta_P, L, Dh, density, v):
    return 2*Dh*delta_P/(L*density*v**2)
    
def plot_coeff_frott():
    L = height
    Dh = channel_width
    density = 755.2
    
    sonde = "SONDE_PRESSION"
    c0, c1 = c0_values[0], c1_values[0]  

    colors = plt.cm.tab10(np.linspace(0, 1, len(porosities)))
    plt.figure(figsize=(15,10), dpi=400)
    plt.grid()
        
    plt.title(rf"$\Lambda$ for $c_0={c0:.1g}, c_1={c1:.1g}$")
    for i, phi in enumerate(porosities):
        #print(phi)
        # File paths

        path_ref_P = os.path.join(f"{run.BUILD_DIRECTORY}/reference/phi_{phi:.1g}/", f"jdd_ref_{sonde}.son")
        path_val_P = os.path.join(f"{run.BUILD_DIRECTORY}/validation/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}/", f"jdd_{sonde}.son")
    
        Y, P_ref = GetSonData(path_ref_P)
        _, P_val = GetSonData(path_val_P)

        P_ref = P_ref[:, 0]
        P_val = P_val[:, 0]
        
        delta_P_ref = np.abs(P_ref[0]- P_ref)
        delta_P_val = np.abs(P_val[0]- P_val)

        path_ref_V = os.path.join(f"{run.BUILD_DIRECTORY}/reference/phi_{phi:.1g}/", f"jdd_ref_SONDE_V_VERTICALE.son")
        path_val_V = os.path.join(f"{run.BUILD_DIRECTORY}/validation/c0_{c0:.1g}_c1_{c1:.1g}/phi_{phi:.1g}/", f"jdd_SONDE_V_VERTICALE.son")
        
        _, data_ref = GetSonData(path_ref_V)
        _, data_val = GetSonData(path_val_V)

        # compute the norm i.e. sqrt(Vx^2 + Vy^2)
        V_ref = np.linalg.norm(data_ref.reshape(-1, 2), axis=1)
        V_val = np.linalg.norm(data_val.reshape(-1, 2), axis=1)

        coef_frott_ref = coeff_frott(delta_P_ref, L, Dh, density, V_ref)
        coef_frott_val = coeff_frott(delta_P_val, L, Dh, density, V_val)

        plt.plot(Y, coef_frott_ref ,marker='*', color = colors[i], label=rf"$\phi_{{ref}}={phi:.1g}$")
        plt.plot(Y, coef_frott_val ,":", color = colors[i], label=rf"$\phi_{{vald}}={phi:.1g}$")

        plt.xlabel('Y [m]')
        plt.ylabel(rf"$\Lambda$")
        
        #plt.ticklabel_format(useOffset=False,style='plain')
        
    plt.legend()
    #plt.tight_layout()
    #plt.savefig("coeff_frott.jpg", dpi=400)
    plt.show()
plot_coeff_frott()

# Error Evaluation

To evaluate the difference between the **validation model** and the **reference solution**, we compute dimensionless error metrics based on normalized L-norms.

## Normalized $L_1$ Norm (Mean Relative Error)

This metric quantifies the **average** relative error across all data points.

$$
L_1 = \frac{1}{n} \sum_{i=1}^{n} \left| \frac{\text{val}_i - \text{ref}_i}{\text{ref}_i} \right|
$$

## Normalized $L_2$ Norm (Root Mean Square Relative Error)

This norm emphasizes **larger deviations**, i.e. outliers.

$$
L_2 = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} \left( \frac{\text{val}_i - \text{ref}_i}{\text{ref}_i} \right)^2 }
$$

## Normalized $L_\infty$ Norm (Maximum Relative Error)

This gives the **worst-case relative error**, i.e. largest deviation.

$$
L_\infty = \max_{i} \left| \frac{\text{val}_i - \text{ref}_i}{\text{ref}_i} \right|
$$
